# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading and exploring a dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, focusing on FAIR-compliant, machine-actionable data packages.

### Dataset Source
The dataset source is provided as a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load Croissant dataset metadata and records from the schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show key info from metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Version: {meta.version}\nIdentifier: {meta.identifier}\nPublished: {meta.datePublished}")

## 2. Data Overview
Review available record sets, their fields, and `@id`s. All entities are referenced by their `@id`, providing stable and FAIR-compliant data access.

In [ ]:
# List all available record sets with their @id and field ids
record_sets = dataset.record_sets
print(f"Discovered {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"- Record Set: {rs['@id']} (name: {rs.get('name', None)})")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for field in fields:
            if isinstance(field, dict):
                print(f"    - Field: {field.get('@id', str(field))} (name: {field.get('name', None)})")
            else:
                print(f"    - Field: {field}")

### Inspect Initial Records from Each Record Set
Let's print a few records from each discovered record set (`@id` is used for reference).

In [ ]:
# Print a sample of the records in each record set, referenced by @id
for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nSample records from record set {rs_id}:")
    try:
        recs = dataset.records(record_set=rs_id)
        for i, rec in enumerate(recs):
            print(rec)
            if i >= 2:
                break
    except Exception as e:
        print(f"  Could not load records: {e}")

## 3. Data Extraction
Load all record sets into dataframes for further analysis. Reference is by record set `@id` and field `@id`, matching the Croissant FAIR model.

In [ ]:
# List of all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    try:
        recs = list(dataset.records(record_set=rs_id))
        if len(recs) > 0:
            dataframes[rs_id] = pd.DataFrame(recs)
            print(f"Loaded DataFrame for record set {rs_id}: shape {dataframes[rs_id].shape}")
        else:
            print(f"No records found for {rs_id}.")
    except Exception as e:
        print(f"Could not load {rs_id}: {e}")

### Inspect a DataFrame
Pick one of the record sets with loaded data and inspect its columns and first rows. Use `@id` at every step.

In [ ]:
# Select the first populated data frame for demonstration
main_record_set_id = None
for rs_id, df in dataframes.items():
    if df.shape[0] > 0:
        main_record_set_id = rs_id
        break
if main_record_set_id:
    print(f"Main dataframe loaded from record set: {main_record_set_id}")
    print("Columns (@id):", df.columns.tolist())
    display(df.head())
else:
    print("No dataframes with data found!")

## 4. Exploratory Data Analysis (EDA)
Common exploratory steps: filter numeric fields, normalize, and (if possible) group by categorical fields. Fields are always referenced by their `@id` in accordance with Croissant best practice.

In [ ]:
import numpy as np
pd.set_option('display.max_columns', 50)

# For demonstration, pick a numeric field and a group field by scanning the column dtypes
df = dataframes[main_record_set_id]
numeric_field_id = None
group_field_id = None

# Guess column types based on dtype
# In actual use, you should look up the field '@id' in the record_set schema
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id:
        group_field_id = col
        break

if numeric_field_id:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().sum() > 0 else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() > 0 else 1)
    print(f"\nNormalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())
    
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize numeric and categorical distributions using the fields referenced by `@id`. (Visualization will run only if there is suitable numeric/categorical data.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field histogram
if numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
- In this notebook, we loaded a FAIR dataset via its Croissant JSON-LD schema using the `mlcroissant` library.
- All data structures and entities were referenced using their `@id`, ensuring stable, reproducible access.
- Record sets and fields were programmatically inspected. Numeric field distributions were analyzed and visualized via `@id`.
- The approach can be extended for full model input/output pipelines while retaining machine-actionable references for data lineage, processing, and reproducibility in ML workflows.

**Next steps:** Map variable definitions directly via Croissant field metadata, link field provenance, and formalize data transformations as Croissant steps.